# MedGemma 1.5 4B — Colab GPU Runner (Hybride)

Lanceur minimal : le code source reste versionné dans le repo Git ; Colab sert uniquement de banc GPU (**L4** pour la baseline, **A100 40 Go** pour QLoRA + évaluation post-fine-tuning). Pensez à **déconnecter le runtime** dès la fin du run.

In [ ]:
# 1. Vérification GPU (bfloat16 matériel SM >= 8.0 requis) & montage Google Drive
import torch
from google.colab import drive

assert torch.cuda.is_available(), 'Aucun GPU détecté : choisir L4 ou A100.'
gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
major, minor = torch.cuda.get_device_capability(0)
print(f'GPU: {gpu_name} ({vram_gb:.1f} Go VRAM) | Compute Capability: {major}.{minor}')
assert major >= 8, (
    f'GPU {gpu_name} (SM {major}.{minor}) sans bfloat16 matériel natif : '
    'passez sur L4 ou A100 dans Exécution > Modifier le type d\'exécution.'
)

drive.mount('/content/drive')
DRIVE_ROOT = '/content/drive/MyDrive/MedGemma_Dermato'

In [ ]:
# 2. Authentification Hugging Face via Colab Secrets (jamais de token en dur)
from google.colab import userdata
from huggingface_hub import login

hf_token = userdata.get('HF_TOKEN')
assert hf_token, 'Ajouter HF_TOKEN dans les Colab Secrets (icône Clé à gauche).'
login(token=hf_token, add_to_git_credential=False)
print('Authentification Hugging Face OK.')

In [ ]:
# 3. Sync GitHub, installation GPU, sys.path et helper de streaming
import os
import subprocess
import sys

REPO_URL = 'https://github.com/sailwaelan-dotcom/medgemma-dermato.git'
REPO_DIR = '/content/medgemma-dermato'
if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'], check=True)

os.chdir(REPO_DIR)
subprocess.run(['pip', 'install', '-q', '-e', '.[train]'], check=True)
if f'{REPO_DIR}/src' not in sys.path:
    sys.path.insert(0, f'{REPO_DIR}/src')

def run_stream(cmd: list[str]) -> None:
    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end='', flush=True)
    rc = proc.wait()
    if rc != 0:
        raise RuntimeError(f'Commande échouée (code {rc}): {" ".join(cmd)}')

import medgemma_dermato.data.isic_ingest  # noqa: F401
print('Package medgemma_dermato prêt.')

In [ ]:
# 4. Baseline Direct (triage, Thinking OFF) sur le hold-out MSKCC 2020 complet (n=200)
import os

DATA_DIR = '/content/drive/MyDrive/MedGemma_Dermato/data/isic_v1'
if not os.path.exists(f'{DATA_DIR}/test.jsonl'):
    run_stream([
        'python', '-u', 'scripts/build_dataset.py',
        '--mode', 'cross-hospital',
        '--commercial-only',
        '--test-stratified-size', '200',
        '--skip-exclude-fetch',
        '--output-dir', DATA_DIR,
    ])

run_stream([
    'python', '-u', 'scripts/run_baseline.py',
    '--manifest', f'{DATA_DIR}/test.jsonl',
    '--image-dir', '/content/drive/MyDrive/MedGemma_Dermato/data/images',
    '--output-dir', '/content/drive/MyDrive/MedGemma_Dermato/outputs/v1_baseline_direct',
    '--run-name', 'medgemma-1.5-4b-baseline-direct',
    '--task', 'triage',
    '--disable-thinking',
    '--max-new-tokens', '384',
])

In [ ]:
# 5. Fine-tuning QLoRA sur HIBA 251 complet (A100 40 Go recommandé, checkpoints Drive)
DATA_DIR = '/content/drive/MyDrive/MedGemma_Dermato/data/isic_v1'

run_stream([
    'python', '-u', 'scripts/train_qlora.py',
    '--train-manifest', f'{DATA_DIR}/train.jsonl',
    '--val-manifest', f'{DATA_DIR}/val.jsonl',
    '--image-dir', '/content/drive/MyDrive/MedGemma_Dermato/data/images',
    '--output-dir', '/content/drive/MyDrive/MedGemma_Dermato/checkpoints/qlora_v1',
])

In [ ]:
# 6. Évaluation Post-QLoRA sur le MÊME hold-out MSKCC (n=200) + Tableau comparatif
from pathlib import Path
from medgemma_dermato.eval.harness import EvalReport, compare_reports

DATA_DIR = '/content/drive/MyDrive/MedGemma_Dermato/data/isic_v1'
OUT_ROOT = Path('/content/drive/MyDrive/MedGemma_Dermato/outputs')
ADAPTER_DIR = '/content/drive/MyDrive/MedGemma_Dermato/checkpoints/qlora_v1/final_adapter'

run_stream([
    'python', '-u', 'scripts/run_baseline.py',
    '--manifest', f'{DATA_DIR}/test.jsonl',
    '--image-dir', '/content/drive/MyDrive/MedGemma_Dermato/data/images',
    '--output-dir', str(OUT_ROOT / 'v1_qlora_bf16'),
    '--adapter-path', ADAPTER_DIR,
    '--run-name', 'medgemma-1.5-4b-qlora-hiba',
    '--task', 'triage',
    '--disable-thinking',
    '--max-new-tokens', '384',
])

rep_base = EvalReport.model_validate_json((OUT_ROOT / 'v1_baseline_direct/eval_report.json').read_text())
rep_qlora = EvalReport.model_validate_json((OUT_ROOT / 'v1_qlora_bf16/eval_report.json').read_text())
comparison = compare_reports([rep_base, rep_qlora], min_malignant_sensitivity=0.95)
(OUT_ROOT / 'sprint3_comparison.md').write_text(comparison + '\n', encoding='utf-8')
print(comparison)

# IMPORTANT : déconnecter le runtime immédiatement après la fin du run :
# from google.colab import runtime; runtime.unassign()